# Seance 8 — Bivariee I : scatterplot et ligne (moyennes)

Voir user guide :
- Marks (2/3) : https://altair-viz.github.io/user_guide/marks/index.html
- Layered & Multi-View Charts : https://altair-viz.github.io/user_guide/compound_charts.html
- Saving Altair Charts : https://altair-viz.github.io/user_guide/saving_charts.html

## Objectifs
- Construire un **scatterplot** pour explorer une relation
- Resumer avec une **ligne de moyennes** (pandas)
- Regler les **axes / echelles** pour etre lisible

Rappel : dans ces seances, on fait les calculs (moyennes, comptages) avec **pandas**.

In [ ]:
import pandas as pd
import altair as alt

alt.data_transformers.disable_max_rows()

data_url = "https://raw.githubusercontent.com/datamisc/ts-2024/main/data.csv"
df = pd.read_csv(data_url, compression="gzip", low_memory=False)

## 1) Scatterplot : polarisation (thermometres)

Variables :
- `V241156` : thermometre Harris (0-100)
- `V241157` : thermometre Trump (0-100)

Chaque point = 1 repondant.

In [ ]:
df_scatter = df.loc[
    df['V241156'].between(0, 100) & df['V241157'].between(0, 100),
    ['V241156', 'V241157']
].copy()

df_scatter.shape

In [ ]:
alt.Chart(df_scatter.sample(2000, random_state=1)).mark_circle(size=25, opacity=0.35).encode(
    x=alt.X('V241157', type='quantitative', title='Thermometre Trump', scale=alt.Scale(domain=[0, 100])),
    y=alt.Y('V241156', type='quantitative', title='Thermometre Harris', scale=alt.Scale(domain=[0, 100]))
).properties(
    title=alt.TitleParams(
        text="Evaluations de Harris et Trump (thermometres 0-100)",
        subtitle=[
            "On observe une relation negative : aimer Trump va souvent avec moins aimer Harris.",
            "Source : ANES 2024 Time Series Study"
        ],
        anchor='start'
    ),
    width=520,
    height=420
)

### Hack-Time 1 (10 min)

Ajoutez une couleur selon l'education (`V241200`) en creant d'abord un sous-tableau `df_scatter2` avec :
- thermometres valides
- education valide (`V241200` > 0)

Puis :
- color = education (type O)
- gardez l'echelle 0-100

In [ ]:
# Hack-Time 1 : votre code ici

## 2) Ligne : moyennes par groupe (ideologie)

Question de comportement politique :
- Les evaluations moyennes de Harris changent-elles selon l'ideologie ?

Variables :
- `V241177` : ideologie (1-7)
- `V241156` : thermometre Harris (0-100)

On calcule les moyennes avec pandas puis on trace une ligne.

In [ ]:
df_mean = df.loc[
    df['V241177'].between(1, 7) & df['V241156'].between(0, 100),
    ['V241177', 'V241156']
].copy()

means = (
    df_mean
    .groupby('V241177', as_index=False)['V241156']
    .mean()
    .rename(columns={'V241177': 'ideologie', 'V241156': 'thermo_harris_moy'})
)

means

In [ ]:
alt.Chart(means).mark_line(point=True, size=3, color='#1D4ED8').encode(
    x=alt.X('ideologie', type='ordinal', title='Ideologie (1-7)'),
    y=alt.Y('thermo_harris_moy', type='quantitative', title='Thermometre Harris (moyenne)', scale=alt.Scale(domain=[0, 100]))
).properties(
    title=alt.TitleParams(
        text="Evaluation moyenne de Harris selon l'ideologie",
        subtitle=[
            "Plus on va vers la droite, plus l'evaluation moyenne de Harris baisse.",
            "Source : ANES 2024 Time Series Study"
        ],
        anchor='start'
    ),
    width=520,
    height=300
)

### Hack-Time 2 (10 min)

Reproduisez le meme graphique mais pour Trump (`V241157`).
- calculez `thermo_trump_moy` avec pandas
- tracez une ligne (echelle 0-100)
- ecrivez un titre + sous-titre

In [ ]:
# Hack-Time 2 : votre code ici

## 3) Inegalites (statut) : education -> revenu

Question : les niveaux d'education sont-ils associes au revenu ?

Variables :
- `V241200` : education (1-5)
- `V242025` : revenu (0-7, categories)

Ici, on trace la moyenne du revenu (code) par education (code).

In [ ]:
df_inc = df.loc[(df['V241200'] > 0) & (df['V242025'] >= 0), ['V241200', 'V242025']].copy()
inc_means = (
    df_inc
    .groupby('V241200', as_index=False)['V242025']
    .mean()
    .rename(columns={'V241200': 'education', 'V242025': 'revenu_moy_code'})
)

alt.Chart(inc_means).mark_line(point=True, size=3, color='#059669').encode(
    x=alt.X('education', type='ordinal', title='Education (code 1-5)'),
    y=alt.Y('revenu_moy_code', type='quantitative', title='Revenu (moyenne du code)')
).properties(
    title=alt.TitleParams(
        text="Education et revenu (codes)",
        subtitle=[
            "Le revenu moyen (code) augmente avec l'education.",
            "Source : ANES 2024 Time Series Study"
        ],
        anchor='start'
    ),
    width=520,
    height=300
)

### Hack-Time 3 (15 min)

Construisez une visualisation alternative de education -> revenu :
- soit un barplot de `revenu_moy_code` par education
- soit un boxplot custom du revenu par education (bonus)

Ajoutez : titre + sous-titre + source.

In [ ]:
# Hack-Time 3 : votre code ici